# 第8回　データの加工：並べ替え・ランキング
## ―― ランキングは何を見せて、何を隠すか

情報活用　／　北星学園大学　2026年度後期

ランキングは、いちばん作りたくなる表である。そして、いちばん誤解を生む表でもある。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

def _build_from_source(keep_missing_code=False):
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    out = out.sort_values("学名").reset_index(drop=True)
    return out if keep_missing_code else out.replace(-999, np.nan)

try:
    df = pd.read_csv("https://aonoa68.github.io/joho-katsuyo/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. 科別 体重ランキングを作る

`groupby` で科ごとにまとめ、`sort_values` で並べ替える。

In [ ]:
d = df.dropna(subset=["体重g"]).copy()

# 種数が少なすぎる科は、平均が安定しないので除く
n_by_fam = d["科"].value_counts()
d = d[d["科"].isin(n_by_fam[n_by_fam >= 10].index)]

rank_tbl = (d.groupby("科")["体重g"].median()
              .sort_values(ascending=False)
              .round(0).reset_index())
rank_tbl.index = rank_tbl.index + 1          # 1位から始める
rank_tbl.columns = ["科", "体重の中央値g"]
rank_tbl

順位表ができた。**1位・2位・3位…と並ぶと、等間隔に差があるように見える。**

では、**実際の差**はどれくらいか。隣どうしの倍率を出す。

In [ ]:
v = rank_tbl["体重の中央値g"].values
gap = pd.DataFrame({
    "順位": [f"{i+1}位 → {i+2}位" for i in range(len(v)-1)],
    "科": [f"{rank_tbl['科'][i+1]} → {rank_tbl['科'][i+2]}" for i in range(len(v)-1)],
    "差(g)": (v[:-1] - v[1:]).astype(int),
    "倍率": (v[:-1] / v[1:]).round(2),
})
gap

**順位表では「1つ分」しか違わない隣どうしが、実際には10%しか違わないこともあれば、3倍以上違うこともある。**

1位と2位はほとんど同じ大きさなのに、下位のほうでは隣に行くだけで体重が数倍変わる。**順位表は、この情報を全部消している。**

### 目で見る ―― 順位と、実際の値

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# 左：順位だけを見た場合（等間隔に並ぶ）
ax[0].barh(range(len(rank_tbl), 0, -1), [1]*len(rank_tbl), color="#80cbc4", edgecolor="white")
ax[0].set_yticks(range(len(rank_tbl), 0, -1))
ax[0].set_yticklabels(rank_tbl["科"])
ax[0].set_xticks([])
ax[0].set_xlabel("順位（1位が上）")
ax[0].set_title("順位表の見え方：どれも「1つ分」の差")

# 右：実際の体重
ax[1].barh(range(len(rank_tbl), 0, -1), rank_tbl["体重の中央値g"], color="#ef9a9a", edgecolor="white")
ax[1].set_yticks(range(len(rank_tbl), 0, -1))
ax[1].set_yticklabels(rank_tbl["科"])
ax[1].set_xlabel("体重の中央値（g）")
ax[1].set_title("実際の値：上位3つに集中している")

plt.tight_layout(); plt.show()

> **順位は差の大きさを消す。** 1位と2位が僅差でも、順位表では等間隔に見える。
> 順位表を作ったら、**必ず実数も併記する。**

---
## 2. 何件から出した順位なのか

第6回でやったとおり、**件数が少ないほど平均はぶれる。**いまのランキングは「種数10以上の科」に絞っていた。**絞らないとどうなるか**を見る。

In [ ]:
# 絞らずに全部の科でランキングを作る
all_fam = (df.dropna(subset=["体重g"])
             .groupby("科")["体重g"]
             .agg(種数="count", 中央値="median")
             .sort_values("中央値", ascending=False)
             .round(0))
all_fam

**1位はヒト科。** 当然に見えるが、**記録があるのは7種だけ**である。
そして下のほうには、**種数1**の科が混ざっている。1種しかない科の「中央値」は、その1種の値そのものだ。

順位表だけを見せられたら、それが7種の話なのか1種の話なのかは分からない。

> **対処は難しくない。表に件数（n）を必ず併記する。** 読む人が自分で判断できるようになる。

In [ ]:
# 少ない件数だと、順位がどれくらい入れ替わるか
w = df["体重g"].dropna()
print("3種だけ取り出して中央値を出す（8回くり返す）")
for i in range(8):
    print(f"  {i+1}回目: {w.sample(3).median():>9,.0f} g")
print()
print(f"（265種すべての中央値は {w.median():,.0f} g）")

**同じ集団から取り出しているのに、桁が変わるほどぶれる。**

あなたの調査でも、選択肢によっては回答が数人しかない項目が出る。そこに順位をつけて「◯◯が1位」と書くと、**偶然を発見として報告することになる。**

---
## 3. 並べ替えと順位付けの書き方

| 関数 | 何をするか |
|---|---|
| `sort_values()` | 行を並べ替える |
| `rank()` | 順位の数字（1, 2, 3…）をつける |

In [ ]:
# 体重が重い順に、上位10種
df.nlargest(10, "体重g")[["学名", "科", "体重g", "集団サイズ"]]

In [ ]:
# 複数のキーで並べ替える（科ごとに、重い順）
(df.dropna(subset=["体重g"])
   .sort_values(["科", "体重g"], ascending=[True, False])
   .head(8)[["学名", "科", "体重g"]])

In [ ]:
# 順位の数字をつける
tmp = df.dropna(subset=["体重g"])[["学名", "科", "体重g"]].copy()
tmp["全体順位"] = tmp["体重g"].rank(ascending=False, method="min").astype(int)
tmp["科内順位"] = tmp.groupby("科")["体重g"].rank(ascending=False, method="min").astype(int)
tmp.sort_values("全体順位").head(10)

`method="min"` は同じ値のときの扱い。同値なら同じ順位（1位が2人なら次は3位）になる。
**同値が多いデータで順位をつけると、順位はほとんど意味を持たなくなる。**
アンケートの5段階評価などは同値だらけになるので、とくに注意する。

In [ ]:
# 件数を併記した表（この形で報告書に載せる）
safe = (df.dropna(subset=["体重g"])
          .groupby("科")["体重g"]
          .agg(種数="count", 中央値="median", 平均="mean", 標準偏差="std")
          .round(0)
          .sort_values("中央値", ascending=False))
safe

---
## 4. 自分の調査データでやる

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# ランキングを作る（列名を書き換える）
# (mydf.groupby("グループの列")["数値の列"]
#      .agg(件数="count", 平均="mean", 中央値="median", 標準偏差="std")
#      .round(1)
#      .sort_values("平均", ascending=False))

---
## 5. 卒業 ―― ランキングを使わずに同じ主張ができるか

作ったランキングで言いたいことを、**順位を使わずに**言い直してみる。
実数で示す、グラフで示す、差の大きさを他の差と比べる。

言い直せたなら、そのランキングは主張の役に立っている。言い直せないなら、**順位という見せ方だけが主張を作っていた**ということになる。

---
## 課題8（6点）

**このノートブック** ＋ **ランキング表** ＋ **「この順位表では言えないこと」3つ**。

- [ ] 自分のデータのランキング表（**件数を併記すること**）
- [ ] この順位表では言えないこと、3つ
- [ ] （書ければ）順位を使わずに同じ主張をした場合の書き方

「言えないこと」の例：1位と2位の差が意味のある差か／件数が少ない項目が混ざっていないか／他の要因で分けたら順位が変わらないか（第7回）

提出期限：次回授業の開始まで（遅れた場合は50%）

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.